<a href="https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one piece of content, for one client, on one day, over 2026-03-01 to 2026-03-31.

In [22]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

In [23]:
import os
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [24]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [32]:
import pandas as pd

HF_BASE = "hf://datasets/FlyRank/internship-warehouse"

fact_cols = [
    "report_date", "client_hash_id", "content_hash_id",
    "gsc_data_available", "ga4_data_available",
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_pageviews", "ga4_sessions", "ga4_engaged_sessions",
    "ga4_total_engagement_sec"
]

panel_daily = pd.read_parquet(
    f"{HF_BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet",
    columns=fact_cols, storage_options={"token": HF_TOKEN}
)

dim_cols = ["client_hash_id", "content_hash_id", "content_type"]
dim_content = pd.read_parquet(
    f"{HF_BASE}/dim_content.parquet",
    columns=dim_cols, storage_options={"token": HF_TOKEN}
)

for col in ["client_hash_id", "content_hash_id"]:
    panel_daily[col] = panel_daily[col].astype("category")
    dim_content[col] = dim_content[col].astype("category")

dim_content_clean = dim_content.drop_duplicates(subset=["client_hash_id","content_hash_id"], keep="first")

panel_daily = panel_daily.merge(dim_content_clean, on=["client_hash_id","content_hash_id"], how="left")

print(panel_daily.columns.tolist())
print(panel_daily.shape)

['report_date', 'client_hash_id', 'content_hash_id', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'content_type']
(9841378, 13)


In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_cols = ["gsc_impressions","gsc_clicks","gsc_avg_position","ga4_engaged_sessions","content_type"]
context_cols = ["client_hash_id","content_hash_id","report_date"]
excluded_cols = ["sessions_ai","ai_chatgpt","ai_perplexity","ai_gemini","ai_copilot","ai_claude","ai_meta","ai_other"]
print("Feature fields:", feature_cols)
print("Context fields:", context_cols)
print("Excluded fields:", excluded_cols)

Feature fields: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_engaged_sessions', 'content_type']
Context fields: ['client_hash_id', 'content_hash_id', 'report_date']
Excluded fields: ['sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature:**

gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_engaged_sessions, ga4_total_engagement_sec, content_type, are observed page-day signals for clustering.

**Label:**

none. Clustering is unsupervised.

**Context:**

client_hash_id, content_hash_id, report_date, are identifiers used to trace results back to real content, not used as clustering inputs.

**Excluded:**

AI-referral columns — under 0.1% row coverage this month (verified in Section 3), not usable yet.
fact_content_daily_performance_sample.parquet — sealed final-month test data, never used for logic.
dim_content date fields (last_optimized_date, content_updated_date, optimization_eligible_date) — could leak information about actions taken after the report window.

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_cols = [
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_pageviews", "ga4_sessions", "ga4_engaged_sessions",
    "ga4_total_engagement_sec", "content_type"
]

label_cols = []  # none — clustering is unsupervised

context_cols = ["client_hash_id", "content_hash_id", "report_date"]

excluded_cols = [
    "sessions_ai", "ai_chatgpt", "ai_perplexity", "ai_gemini",
    "ai_copilot", "ai_claude", "ai_meta", "ai_other"
]

print("Feature fields:", feature_cols)
print("Label fields:", label_cols)
print("Context fields:", context_cols)
print("Excluded fields:", excluded_cols)

Feature fields: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'content_type']
Label fields: []
Context fields: ['client_hash_id', 'content_hash_id', 'report_date']
Excluded fields: ['sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Grain
n_unique = panel_daily.groupby(["client_hash_id","content_hash_id","report_date"], observed=True).ngroups
print("Row count:", len(panel_daily))
print("Unique combos:", n_unique)
print("Duplicates on key?", n_unique != len(panel_daily))

# Row count and date span
print("Date span:", panel_daily["report_date"].min(), "-", panel_daily["report_date"].max())

# Availability
available = panel_daily.query("gsc_data_available == True and ga4_data_available == True")
print("Rows before filter:", len(panel_daily))
print("Rows after IS TRUE filter:", len(available))

# Five features
feature_frame = panel_daily[[
    "client_hash_id","content_hash_id",
    "gsc_impressions","gsc_clicks","gsc_avg_position",
    "ga4_engaged_sessions","content_type"
]].copy()
feature_frame.head()

Row count: 9841378
Unique combos: 9841378
Duplicates on key? False
Date span: 2026-03-01 - 2026-03-31
Rows before filter: 9841378
Rows after IS TRUE filter: 364347


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,content_type
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,keyword article
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,keyword article
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,keyword article
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,keyword article
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,keyword article


In [37]:
# The trap
panel_april = pd.read_parquet(
    f"{HF_BASE}/fact_content_daily_performance/month=2026-04/data_0.parquet",
    columns=["client_hash_id","content_hash_id","gsc_clicks"],
    storage_options={"token": HF_TOKEN}
)
future_clicks = (
    panel_april.groupby(["client_hash_id","content_hash_id"], observed=True)["gsc_clicks"]
    .sum().rename("future_clicks").reset_index()
)

content_level = feature_frame.groupby(["client_hash_id","content_hash_id"], observed=True).agg({
    "gsc_impressions":"sum","gsc_clicks":"sum","gsc_avg_position":"mean",
    "ga4_engaged_sessions":"sum","content_type":"first"
}).reset_index()

leaky = content_level.merge(future_clicks, on=["client_hash_id","content_hash_id"], how="left")
leaky["future_clicks"] = leaky["future_clicks"].fillna(0)

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

X_leaky = StandardScaler().fit_transform(leaky[["gsc_impressions","gsc_clicks","gsc_avg_position","ga4_engaged_sessions","future_clicks"]].fillna(0))
km_leaky = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_leaky)

rng = np.random.default_rng(42)
idx = rng.choice(len(X_leaky), size=min(20000, len(X_leaky)), replace=False)
print("Silhouette WITH leak:", round(silhouette_score(X_leaky[idx], km_leaky.labels_[idx]), 3))

X_honest = StandardScaler().fit_transform(content_level[["gsc_impressions","gsc_clicks","gsc_avg_position","ga4_engaged_sessions"]].fillna(0))
km_honest = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_honest)

idx2 = rng.choice(len(X_honest), size=min(20000, len(X_honest)), replace=False)
print("Silhouette WITHOUT leak (honest):", round(silhouette_score(X_honest[idx2], km_honest.labels_[idx2]), 3))

Silhouette WITH leak: 0.705
Silhouette WITHOUT leak (honest): 0.72


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Filtering to rows where both GSC and GA4 data are available (IS TRUE) keeps only 364,347 of 9,841,378 rows (3.7%). The rest split as: GSC-only 17.5%, GA4-only 0.5%, and neither source available for 47.7% of rows — meaning roughly half of all page-days in this warehouse carry no performance signal at all for March. Clustering on the fully-available slice would represent a small, non-random subset of content.

gsc_avg_position is null for 6,230,317 rows (63%), and this is fully structural: 100% of those null rows have zero impressions, confirming there's no position to average when a page had no search visibility that day — not a data quality gap.

AI-referral columns are populated in fewer than 0.1% of rows this month, confirming they're too sparse to use as features yet.

This slice covers one month (March 2026) across 55 unique clients — seasonal effects or this client mix won't necessarily generalize to other months or a larger client base.

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Row-level availability breakdown
both = panel_daily.query("gsc_data_available == True and ga4_data_available == True")
gsc_only = panel_daily.query("gsc_data_available == True and ga4_data_available == False")
ga4_only = panel_daily.query("gsc_data_available == False and ga4_data_available == True")
neither = panel_daily.query("gsc_data_available == False and ga4_data_available == False")

print("Both available:", len(both), f"({len(both)/len(panel_daily):.1%})")
print("GSC only:", len(gsc_only), f"({len(gsc_only)/len(panel_daily):.1%})")
print("GA4 only:", len(ga4_only), f"({len(ga4_only)/len(panel_daily):.1%})")
print("Neither:", len(neither), f"({len(neither)/len(panel_daily):.1%})")

# gsc_avg_position missingness — check it's structural (tied to zero impressions)
zero_impressions_null_position = panel_daily.query("gsc_impressions == 0")["gsc_avg_position"].isna().sum()
total_null_position = panel_daily["gsc_avg_position"].isna().sum()
print(f"\nNull gsc_avg_position rows: {total_null_position}")
print(f"Of those, rows with 0 impressions: {zero_impressions_null_position} ({zero_impressions_null_position/total_null_position:.1%})")

# reconcile client counts (you had 55 vs 65 from different queries — resolve it here)
print("\nTotal unique clients:", panel_daily["client_hash_id"].nunique())
print("Clients with GSC available (any row):", panel_daily.query("gsc_data_available == True")["client_hash_id"].nunique())
print("Clients with GA4 available (any row):", panel_daily.query("ga4_data_available == True")["client_hash_id"].nunique())

Both available: 364347 (3.7%)
GSC only: 1718348 (17.5%)
GA4 only: 49619 (0.5%)
Neither: 4690323 (47.7%)

Null gsc_avg_position rows: 6230317
Of those, rows with 0 impressions: 6230317 (100.0%)

Total unique clients: 55
Clients with GSC available (any row): 47
Clients with GA4 available (any row): 41


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.